# Step 4 prep: train the siamese model on `case2s`

**Why:** the Step 3 model saw only easy and same-street negatives, so its `case2s` score (in `model_baseline/07_backtest_case2s.ipynb`) can't be compared with the classical models, which were retrained on the new train split. This notebook trains the same model on `case2s`, so the backtest compares like with like.

**What it does:** the same protocol as `13_real_training.ipynb` (character-level BiLSTM encoder, mean pooling, seed 0, Adam 1e-3, up to 20 epochs, patience 3, best epoch = highest val "found" among epochs above the case1 guardrail), now on the `case2s` train and val splits. One run. Checkpoint and history go to `checkpoints_case2s/` (git-ignored), same file names as `checkpoints/`, so `load_run(0, "mean", "checkpoints_case2s")` loads it.

**Time:** about 4 min per epoch on the old train split; the `case2s` train split has the same number of rows. Worst case 20 epochs ≈ 80 min. The run is skipped if the checkpoint already exists.

**Decision for you:** the early-stopping rule only looks at non-identical matches, case1 and easy negatives, not at `case2s`. Kept identical to Step 3 so the results are comparable; changing it (for example adding a `case2s` guardrail) is your call.

## 0. Setup
**Why:** imports, constants and the checkpoint folder.

In [1]:
# Setup: imports and constants
%load_ext autoreload
%autoreload 2

import os
import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd
import torch
from torch.utils.data import DataLoader

from helpers.pairs import load_pairs

# digit-prefixed filenames -> can't use `from ... import`
build_vocab = import_module("siamese_model.01_vocab_step").build_vocab
_data_step = import_module("siamese_model.11_data_plumbing")
PairDataset, make_collate = _data_step.PairDataset, _data_step.make_collate
_real_step = import_module("siamese_model.13_real_training")
train_run, broken_run_problems, load_run, rebuild_model = _real_step.train_run, _real_step.broken_run_problems, _real_step.load_run, _real_step.rebuild_model

DATASET = "overture_case2s"
CHECKPOINT_DIR_CASE2S = "checkpoints_case2s"   # git-ignored, relative to siamese_model/
SEED, POOLING = 0, "mean"
BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE = 64, 1e-3, 20, 3
os.makedirs(CHECKPOINT_DIR_CASE2S, exist_ok = True)

pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)

## 1. Load and build the vocabulary
**Why:** the model reads characters, so it needs a character → id table. As in Step 3 it is built from the train split only, so characters seen only in val / test map to UNK.  
**Does:** loads the `case2s` train and val pairs, builds the vocabulary from train, wraps each split in a `DataLoader` (train shuffled, val not).  
**Output:** `vocab`, `train_loader`, `val_loader`

In [2]:
# Load case2s train / val, vocab from train only, loaders
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
vocab = build_vocab(train)
print(len(train), len(val), len(vocab))

train_loader = DataLoader(PairDataset(train, train_label), batch_size = BATCH_SIZE, shuffle = True, collate_fn = make_collate(vocab))
val_loader = DataLoader(PairDataset(val, val_label), batch_size = BATCH_SIZE, shuffle = False, collate_fn = make_collate(vocab))

202568 41865 40


## 2. Train and save
**Why:** one run with the Step 3 settings, kept on disk so the backtest can load it and this cell is skipped on a re-run.  
**Does:** `train_run` (the loop from `13_real_training.ipynb`, as a function) trains until patience runs out, keeping the best epoch's weights; the history and the checkpoint (weights, vocab, config, seed, best row) are saved; `broken_run_problems` lists anything suspicious in the history (`[]` = healthy).  
**Output:** `checkpoints_case2s/overture_siamese_seed_0_pooling_mean.pt`, `history_seed_0_pooling_mean.csv`

In [3]:
# Train (skipped if the checkpoint exists), save history + checkpoint, check the run
ckpt_path = f"{CHECKPOINT_DIR_CASE2S}/overture_siamese_seed_{SEED}_pooling_{POOLING}.pt"
if not os.path.exists(ckpt_path):
    history, best, best_state = train_run(SEED, POOLING, vocab, train_loader, val_loader, val, val_label,
                                          lr = LR, max_epochs = MAX_EPOCHS, patience = PATIENCE)
    history.to_csv(f"{CHECKPOINT_DIR_CASE2S}/history_seed_{SEED}_pooling_{POOLING}.csv", index = False)
    torch.save({
        "state_dict": best_state, "vocab": vocab,
        "config": {"emb_dim": 32, "hidden": 128, "pooling": POOLING, "max_len": 85},
        "seed": SEED, "best": best}, ckpt_path)

history, ckpt = load_run(SEED, POOLING, checkpoint_dir = CHECKPOINT_DIR_CASE2S)
print("best epoch:", ckpt["best"])
print("problems:", broken_run_problems(history))
history

{'epoch': 0, 'train_loss': 0.0660264564348592, 'val_loss': 0.042159002274274826, 'found': 56.46173149309912, 'case1': 99.89115448806328, 'easy': 99.99129261178109}
{'epoch': 1, 'train_loss': 0.03080079882001626, 'val_loss': 0.030916577205061913, 'found': 74.27854454203262, 'case1': 99.47754154270372, 'easy': 99.99129261178109}
{'epoch': 2, 'train_loss': 0.02353198803829014, 'val_loss': 0.030690088868141174, 'found': 77.91718946047679, 'case1': 99.579130687178, 'easy': 99.9825852235622}
{'epoch': 3, 'train_loss': 0.01996379200356283, 'val_loss': 0.02882418781518936, 'found': 76.16060225846925, 'case1': 99.62992525941513, 'easy': 99.98693891767165}
{'epoch': 4, 'train_loss': 0.017089247394668435, 'val_loss': 0.030150331556797028, 'found': 78.92095357590966, 'case1': 99.31790145852986, 'easy': 99.96517044712438}
{'epoch': 5, 'train_loss': 0.01543299426856201, 'val_loss': 0.03001861274242401, 'found': 79.92471769134254, 'case1': 99.579130687178, 'easy': 99.9825852235622}
{'epoch': 6, 'trai

,epoch,train_loss,val_loss,found,case1,easy
0,0,0.066026,0.042159,56.461731,99.891154,99.991293
1,1,0.030801,0.030917,74.278545,99.477542,99.991293
2,2,0.023532,0.030690,77.917189,99.579131,99.982585
3,3,0.019964,0.028824,76.160602,99.629925,99.986939
4,4,0.017089,0.030150,78.920954,99.317901,99.965170
5,5,0.015433,0.030019,79.924718,99.579131,99.982585
6,6,0.013551,0.029072,79.422836,99.463029,99.978232
7,7,0.012048,0.029410,78.419072,99.448516,99.969524
8,8,0.010608,0.032221,83.061481,99.252594,99.952109
9,9,0.009395,0.031663,80.175659,99.426747,99.978232
